# CardioIA - Fase 2 - Parte 1
## Extração de sintomas e sugestão de diagnóstico

Neste notebook eu leio 10 relatos de pacientes, procuro sintomas conhecidos em cada relato
e sugiro um diagnóstico com base em um mapa de conhecimento.

A lógica mora em `src/extracao_sintomas.py`. O notebook importa de lá em vez de copiar o
código, para não existirem duas versões da mesma função se eu precisar corrigir algo.

**Aviso:** exercício acadêmico. A saída é uma sugestão por busca de palavras, não um
diagnóstico médico.

### 1. Preparação

Aponto o Python para a pasta `src/` e importo o módulo. Uso `inspect.getsource` mais
adiante para mostrar o código das funções principais aqui dentro do notebook.

In [1]:
import sys, inspect
from pathlib import Path

import pandas as pd

# O notebook roda a partir de notebooks/, então a raiz do projeto é a pasta de cima.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import extracao_sintomas as ex

pd.set_option("display.max_colwidth", 90)
print("Raiz do projeto:", RAIZ)

Raiz do projeto: /home/claude/cardioia-fase2


### 2. Normalização do texto

Este é o passo que faz o resto funcionar. Os relatos reais escrevem o mesmo sintoma de
formas diferentes: `coracao` e `coração`, `CANSAÇO CONSTANTE` e `cansaço constante`.
Se eu comparar as strings do jeito que vieram, o computador acha que são coisas distintas.

A função tira o acento, passa para minúsculo e limpa a pontuação, dos dois lados da
comparação (relato e mapa de conhecimento).

In [2]:
print(inspect.getsource(ex.normalizar))

def normalizar(texto):
    """Deixa o texto em um formato único para comparação.

    Faz quatro coisas, nesta ordem:
      1. separa a letra do acento (NFKD: "ç" vira "c" + cedilha solta);
      2. joga fora os acentos soltos;
      3. passa tudo para minúsculo;
      4. troca pontuação por espaço e colapsa espaços repetidos.

    Por que isso importa: os relatos reais têm "coracao" e "coração",
    "CANSAÇO CONSTANTE" e "cansaço constante". Sem normalizar, o mesmo
    sintoma seria tratado como duas coisas diferentes e o sistema erraria.

    >>> normalizar("CANSAÇO  constante!")
    'cansaco constante'
    """
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = texto.lower()
    texto = re.sub(r"[^a-z0-9 ]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()



In [3]:
# Três formas de escrever o mesmo sintoma viram exatamente a mesma string.
exemplos = ["CANSAÇO  constante!", "cansaco constante", "Cansaço Constante"]
for e in exemplos:
    print(f"{e!r:30} -> {ex.normalizar(e)!r}")

'CANSAÇO  constante!'          -> 'cansaco constante'
'cansaco constante'            -> 'cansaco constante'
'Cansaço Constante'            -> 'cansaco constante'


### 3. Mapa de conhecimento

O arquivo `dados/mapa_conhecimento.csv` liga expressões de sintoma a doenças. Cada linha
tem duas expressões equivalentes (`sintoma_1` e `sintoma_2`) que apontam para a mesma
doença, mais duas colunas que eu acrescentei além do que o enunciado pedia:

- **`peso`** (1 a 3): nem todo sintoma vale o mesmo. "Dor no peito" é inespecífica;
  "irradiou para o braço esquerdo" é muito mais sugestiva de infarto.
- **`origem`**: de onde veio o termo. `texto_01` e `texto_02` são os corpora do SciELO
  que o grupo preparou na Fase 1; `conhecimento_geral_equipe` é o que nós escrevemos sem
  respaldo direto nesses textos. Sem essa coluna, o mapa pareceria todo fundamentado em
  literatura científica, quando boa parte dele é julgamento nosso.

In [4]:
mapa_bruto = pd.read_csv(RAIZ / "dados" / "mapa_conhecimento.csv")
print(f"{len(mapa_bruto)} linhas no mapa")
mapa_bruto.head(8)

48 linhas no mapa


,sintoma_1,sintoma_2,doenca_associada,peso,origem
0,dor no peito,dor torácica,Infarto Agudo do Miocárdio,2,texto_02_sindrome_coronariana_scielo
1,aperto no tórax,peso no peito,Infarto Agudo do Miocárdio,2,conhecimento_geral_equipe
2,irradiou para o braço esquerdo,dor no braço esquerdo,Infarto Agudo do Miocárdio,3,texto_02_sindrome_coronariana_scielo
3,suor frio,sudorese,Infarto Agudo do Miocárdio,2,texto_02_sindrome_coronariana_scielo
4,enjoo,náusea,Infarto Agudo do Miocárdio,1,texto_02_sindrome_coronariana_scielo
5,dor no peito em repouso,dor que não passa parado,Infarto Agudo do Miocárdio,3,conhecimento_geral_equipe
6,sensação de morte iminente,angústia no peito,Infarto Agudo do Miocárdio,2,conhecimento_geral_equipe
7,dor muito forte,dor insuportável,Infarto Agudo do Miocárdio,1,conhecimento_geral_equipe


In [5]:
# Quanto do mapa tem respaldo nos textos da Fase 1 e quanto é conhecimento nosso.
mapa_bruto["origem"].value_counts()

origem
conhecimento_geral_equipe               32
texto_02_sindrome_coronariana_scielo    15
texto_01_hipertensao_scielo              1
Name: count, dtype: int64

In [6]:
# carregar_mapa achata as duas colunas de sintoma em uma lista única de termos,
# já normalizados e ordenados do mais longo para o mais curto.
termos = ex.carregar_mapa()
print(f"{len(termos)} expressões prontas para busca")
termos[0]

96 expressões prontas para busca


{'expressao': 'episódio curto que passa sozinho',
 'expressao_norm': 'episodio curto que passa sozinho',
 'doenca': 'Arritmia',
 'peso': 2,
 'origem': 'conhecimento_geral_equipe'}

### 4. Relatos dos pacientes

São 10 frases em `dados/frases_pacientes.txt`. Escrevi de propósito algumas sem
acentuação e uma parcialmente em caixa alta, para testar a normalização com o tipo de
variação que aparece em texto digitado por gente de verdade.

In [7]:
relatos = ex.carregar_frases()
for i, r in enumerate(relatos, 1):
    print(f"{i:02d}. {r}")

01. Há dois dias sinto um aperto no tórax que aparece quando subo a escada do prédio e some assim que eu paro para descansar; já deixei de levar meu filho a pé na escola por causa disso.
02. Faz uma semana que acordo com CANSAÇO CONSTANTE, mesmo dormindo oito horas por noite, e no fim da tarde não consigo mais terminar o serviço.
03. Desde ontem a noite estou com falta de ar quando deito, preciso empilhar dois travesseiros para conseguir dormir e acordo várias vezes.
04. Sinto o coracao disparado do nada, umas tres vezes por semana faz um mes, dura uns minutos e passa sozinho; parei de dirigir com medo de acontecer no transito.
05. Tenho dor de cabeça na nuca quase todas as manhãs há uns vinte dias, junto com a visão embaçada, e tive que parar de ler no computador no trabalho.
06. Minhas pernas estão inchando no fim do dia faz duas semanas, o sapato aperta e eu preciso deitar com o pé para cima para melhorar.
07. Ontem tive uma dor no peito muito forte que irradiou para o braço esquerd

### 5. Extração de sintomas de um relato

A busca é por trecho: eu pergunto se a expressão normalizada aparece dentro do relato
normalizado.

Tem uma regra a mais, contra contagem dupla. Quando um termo longo casa com um pedaço do
relato, aquele pedaço fica ocupado; um termo curto que caia inteiro dentro dele é
descartado. Sem isso, no relato 3 tanto `falta de ar quando deito` (peso 3) quanto
`falta de ar` (peso 2) contariam, e a Insuficiência Cardíaca somaria 5 onde deveria somar
3. O número ficaria maior sem evidência nova nenhuma por trás.

In [8]:
print(inspect.getsource(ex.extrair_sintomas))

def extrair_sintomas(relato, termos):
    """Procura no relato todas as expressões do mapa de conhecimento.

    Regra anti-contagem-dupla: quando um termo longo casa com um trecho do
    relato, aquele trecho fica "ocupado". Um termo mais curto que caia inteiro
    dentro de um trecho já ocupado é descartado.

    Exemplo: no relato "estou com falta de ar quando deito", o termo
    "falta de ar quando deito" (peso 3) casa primeiro. O termo "falta de ar"
    (peso 2) cairia dentro dele, então é ignorado. Sem essa regra, a
    Insuficiência Cardíaca somaria 5 pontos onde deveria somar 3, e a
    confiança do sistema ficaria artificialmente inflada.
    """
    relato_norm = normalizar(relato)
    trechos_ocupados = []  # lista de pares (inicio, fim)
    achados = []

    for termo in termos:
        posicao = relato_norm.find(termo["expressao_norm"])
        if posicao == -1:
            continue

        inicio, fim = posicao, posicao + len(termo["expressao_norm"])
        dentro_de_ou

In [9]:
# Relato 3 é o caso da regra anti-contagem-dupla.
achados = ex.extrair_sintomas(relatos[2], termos)
print(relatos[2], "\n")
for t in achados:
    print(f"  {t['expressao']:30} -> {t['doenca']:28} peso {t['peso']}  [{t['origem']}]")

Desde ontem a noite estou com falta de ar quando deito, preciso empilhar dois travesseiros para conseguir dormir e acordo várias vezes. 

  falta de ar quando deito       -> Insuficiência Cardíaca       peso 3  [conhecimento_geral_equipe]
  dois travesseiros              -> Insuficiência Cardíaca       peso 3  [conhecimento_geral_equipe]


### 6. Sugestão de diagnóstico

Somo os pesos por doença e devolvo a mais pontuada. Dois cuidados:

- se nenhum termo casar, a saída é **inconclusivo** em vez de um chute;
- se duas doenças empatarem na frente, também é **inconclusivo** — preferimos devolver
  "não sei" a devolver uma moeda lançada.

A `confianca` é a fatia da evidência que aponta para a vencedora, não uma probabilidade.
100% quer dizer "todos os sintomas que eu achei apontam para a mesma doença", e não
"tenho certeza do diagnóstico".

In [10]:
print(inspect.getsource(ex.sugerir_diagnostico))

def sugerir_diagnostico(achados):
    """Soma os pesos por doença e devolve a mais pontuada.

    Devolve um dicionário com:
      diagnostico  -> doença vencedora (ou 'Inconclusivo...')
      pontuacao    -> soma dos pesos da vencedora
      confianca    -> pontuação da vencedora / soma de todas as pontuações
      placar       -> lista completa (doença, pontos), da maior para a menor
      empate       -> True se duas ou mais doenças empataram na frente

    A "confiança" NÃO é probabilidade. É só a fatia da evidência que aponta
    para a doença vencedora. Um valor de 1.0 significa "todos os sintomas
    encontrados apontam para a mesma doença", e não "temos certeza".
    """
    if not achados:
        return {
            "diagnostico": ROTULO_INCONCLUSIVO,
            "pontuacao": 0,
            "confianca": 0.0,
            "placar": [],
            "empate": False,
        }

    placar = defaultdict(int)
    for termo in achados:
        placar[termo["doenca"]] += termo["peso"

In [11]:
resultado = ex.sugerir_diagnostico(achados)
resultado

{'diagnostico': 'Insuficiência Cardíaca',
 'pontuacao': 6,
 'confianca': 1.0,
 'placar': [('Insuficiência Cardíaca', 6)],
 'empate': False}

### 7. Rodando nos 10 relatos

In [12]:
df = ex.analisar_relatos(relatos, termos)
df[["id_relato", "sintomas_encontrados", "diagnostico_sugerido", "confianca", "placar_completo"]]

,id_relato,sintomas_encontrados,diagnostico_sugerido,confianca,placar_completo
0,1,aperto no tórax; subo a escada; paro para descansar,Angina,0.75,Angina:6 | Infarto Agudo do Miocárdio:2
1,2,cansaço constante; mesmo dormindo,Insuficiência Cardíaca,1.00,Insuficiência Cardíaca:4
2,3,falta de ar quando deito; dois travesseiros,Insuficiência Cardíaca,1.00,Insuficiência Cardíaca:6
3,4,coração disparado; dura uns minutos e passa,Arritmia,1.00,Arritmia:5
4,5,dor de cabeça na nuca; visão embaçada,Hipertensão Arterial,1.00,Hipertensão Arterial:4
5,6,inchando; sapato aperta; pé para cima,Insuficiência Cardíaca,1.00,Insuficiência Cardíaca:6
6,7,dor no peito; irradiou para o braço esquerdo; suor frio; enjoo,Infarto Agudo do Miocárdio,1.00,Infarto Agudo do Miocárdio:8
7,8,dor no peito; respiro fundo; me inclino para a frente; dormir de barriga para cima,Pericardite,0.80,Pericardite:8 | Infarto Agudo do Miocárdio:2
8,9,tontura forte; fala ficou enrolada; ficou fraco,Acidente Vascular Cerebral,1.00,Acidente Vascular Cerebral:8
9,10,tosse seca a noite; canso para caminhar,Insuficiência Cardíaca,1.00,Insuficiência Cardíaca:4


In [13]:
df["diagnostico_sugerido"].value_counts()

diagnostico_sugerido
Insuficiência Cardíaca        4
Angina                        1
Arritmia                      1
Hipertensão Arterial          1
Infarto Agudo do Miocárdio    1
Pericardite                   1
Acidente Vascular Cerebral    1
Name: count, dtype: int64

### 8. Teste de robustez

Os 10 relatos acima foram escritos pelo grupo junto com o mapa de conhecimento, então
acertar todos não prova quase nada. Abaixo eu testo com frases que **não** foram usadas
para montar o mapa, incluindo uma sem nenhum sintoma cardíaco e uma ambígua de propósito.

In [14]:
frases_novas = [
    "Torci o tornozelo jogando bola no sábado e está doendo para pisar.",
    "Meu coracao fica acelerado e sinto falta de ar quando subo a escada do trabalho.",
    "Estou com dor de garganta e febre desde segunda-feira.",
    "sinto um PESO NO PEITO ha tres dias, junto com suor frio",
]

for f in frases_novas:
    a = ex.extrair_sintomas(f, termos)
    r = ex.sugerir_diagnostico(a)
    print(f"\n{f}")
    print(f"   sintomas    : {[t['expressao'] for t in a] or '(nenhum)'}")
    print(f"   diagnóstico : {r['diagnostico']}  (confiança {r['confianca']:.0%})")
    print(f"   placar      : {r['placar']}")


Torci o tornozelo jogando bola no sábado e está doendo para pisar.
   sintomas    : (nenhum)
   diagnóstico : Inconclusivo - encaminhar para avaliação humana  (confiança 0%)
   placar      : []

Meu coracao fica acelerado e sinto falta de ar quando subo a escada do trabalho.
   sintomas    : ['falta de ar', 'subo a escada']
   diagnóstico : Angina  (confiança 60%)
   placar      : [('Angina', 3), ('Insuficiência Cardíaca', 2)]

Estou com dor de garganta e febre desde segunda-feira.
   sintomas    : (nenhum)
   diagnóstico : Inconclusivo - encaminhar para avaliação humana  (confiança 0%)
   placar      : []

sinto um PESO NO PEITO ha tres dias, junto com suor frio
   sintomas    : ['peso no peito', 'suor frio']
   diagnóstico : Infarto Agudo do Miocárdio  (confiança 100%)
   placar      : [('Infarto Agudo do Miocárdio', 4)]


**Leia a saída acima com atenção — o caso 2 falhou.**

A frase diz "Meu coracao fica acelerado". O mapa tem o termo `coração acelerado`, mas a
palavra "fica" está no meio, e a busca por trecho exige as palavras coladas. Resultado: o
sinal de arritmia foi perdido, e o sistema respondeu Angina com 60% de confiança, apoiado
só em `falta de ar` e `subo a escada`.

Não é um bug de digitação meu: é o limite do método. Busca por trecho não tolera palavra
intrusa. Corrigir isso exigiria casar por radical de palavra ou por distância entre
termos, o que já sai do escopo de "estrutura simples" que o enunciado pede. Deixo
registrado em vez de esconder.

### 9. Limitações desta parte

O que este código **não** faz, e que eu preciso saber dizer na defesa:

1. **Ele não entende a frase, só procura pedaços de texto.** "Não sinto dor no peito"
   casa com `dor no peito` exatamente como "sinto dor no peito" casaria. Negação,
   ironia e hipótese ("o médico perguntou se eu tinha dor no peito") passam batido.
2. **O mapa e as frases foram escritos pela mesma equipe.** Acertar 10 de 10 nos relatos
   próprios é circular: eu escrevi a pergunta e o gabarito. O teste da seção 8 é mais
   honesto, e lá o sistema já erra e fica inconclusivo em alguns casos.
3. **Os pesos são opinião nossa, não estatística.** Ninguém calibrou esses números
   contra desfecho clínico real. Trocar um peso de 2 para 3 muda diagnóstico.
4. **A hipertensão é a parte mais frágil do mapa.** O `texto_01_hipertensao_scielo` da
   Fase 1 trata a doença como frequentemente assintomática e não lista cefaleia, tontura
   ou visão embaçada. Quase todas as linhas de Hipertensão estão marcadas como
   `conhecimento_geral_equipe` — é crença nossa, não evidência documentada.
5. **Cobertura limitada.** São 7 condições. Qualquer coisa fora disso vira inconclusivo,
   o que é o comportamento certo, mas também mostra o tamanho real do sistema.
6. **Palavra intrusa quebra o casamento.** Como o caso 2 da seção 8 mostrou,
   "coracao fica acelerado" não casa com `coração acelerado`. Em produção isso seria
   grave: um sintoma de arritmia sumiu porque o paciente escreveu uma palavra a mais.

### 10. Salvando o resultado

O CSV gerado aqui alimenta o README e serve de evidência da execução.

In [15]:
saida = RAIZ / "dados" / "resultado_extracao.csv"
df.to_csv(saida, index=False, encoding="utf-8")
print("Arquivo gerado:", saida)
print(f"{len(df)} relatos processados, "
      f"{(df['diagnostico_sugerido'] == ex.ROTULO_INCONCLUSIVO).sum()} inconclusivos.")

Arquivo gerado: /home/claude/cardioia-fase2/dados/resultado_extracao.csv
10 relatos processados, 0 inconclusivos.
